## tl;dr
This notebook audits deterministic engineering fixtures for two neonatal movement IMUs. They are synthetic, not clinical ground truth. Fine tremor is modeled above 6 Hz at 100 Hz; the separate 10 Hz export intentionally demonstrates aliasing.

## Context & Methods
Sources and assumptions are documented in `../synthetic-data/README.md`. The amplitude range is an explicit stress-test assumption because no matching neonatal ICM-20948 amplitude distribution was found.

### Key Assumptions
- 100 Hz source rate; 10 Hz legacy decimation.
- Fine tremor frequency 6.5–8.5 Hz with modulation and drift.
- Datasheet-derived white noise plus slow bias drift.
- Artifact classes are never labeled as infant tremor.

## Data

In [ ]:
from pathlib import Path
import json
root = Path('..')
manifest = json.loads((root / 'synthetic-data' / 'manifest.json').read_text())
[(s['name'], s['expected'], s['metrics_100hz']) for s in manifest['scenarios']]

## Results

In [ ]:
assert manifest['synthetic'] is True
assert manifest['clinical_ground_truth'] is False
assert manifest['sample_rates_hz'] == {'ground_truth': 100, 'legacy_app_view': 10}
by_name = {s['name']: s for s in manifest['scenarios']}
assert by_name['still_threshold']['metrics_100hz']['movement_p99'] < 0.12
assert 6.0 < by_name['fine_tremor_low']['metrics_100hz']['dominant_raw_accel_1_20hz_sensor1'] < 9.0
print('Core provenance, stillness, and tremor-band checks passed.')

## Takeaways
Use the 100 Hz fixtures for tremor feature development. Use the 10 Hz fixtures only to expose current pipeline failures. Validate all parameter distributions against synchronized real IMU plus video annotations before clinical use.